# GATraj-RiskMotion on Google Colab

Choose **Runtime → Change runtime type → GPU**, then run the cells in order. The notebook mounts Google Drive, trains the corrected baseline or proposed motion module, and resumes from the latest checkpoint in Drive after a runtime restart. The full learning-rate schedule remains 1,000 epochs even when training in shorter sessions.

Use the same settings and seeds for baseline and modified models. Improvement in ADE/FDE remains an experimental hypothesis.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

## Clone and install

Code runs from Colab's local disk. Checkpoints, configurations, caches and evaluation reports are written directly to your mounted Drive. Enable a GPU before continuing.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

DRIVE_ROOT = Path("/content/drive/MyDrive")
assert DRIVE_ROOT.is_dir(), "Mount Google Drive using the previous cell."
REPO = Path("/content/GATraj-RiskMotion")
if not REPO.exists():
    subprocess.run(["git", "clone", "https://github.com/Kedar2002/GATraj-RiskMotion.git", str(REPO)], check=True)
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
subprocess.run(["git", "rev-parse", "--short", "HEAD"], check=True)
import torch
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
assert torch.cuda.is_available(), "Select a GPU under Runtime → Change runtime type."
print("GPU:", torch.cuda.get_device_name(0))

## Configure one experiment

Set `MOTION_MODULE=False` for the corrected baseline and `True` for RiskMotion. Keep `FIX_ATTENTION=True` in both arms. Fold IDs: 0 ETH, 1 Hotel, 2 Zara1, 3 Zara2, 4 University. Different module combinations and seeds receive separate experiment directories.

For a startup check, set `STOP_AT_EPOCH=1`, keeping `TOTAL_EPOCHS=1000`. Later restore `STOP_AT_EPOCH=TOTAL_EPOCHS` to continue the same schedule.

Every completed epoch is checkpointed directly to Drive. To conserve storage, after each successful chunk the default cleanup keeps every 100th epoch and the latest checkpoint. Set `PRUNE_INTERMEDIATE_CHECKPOINTS=False` if you need all epochs.

In [ ]:
TEST_SET = 1
SEED = 1
MOTION_MODULE = True
RELATIVE_VELOCITY = True
CLOSEST_APPROACH = True
CONFIDENCE_GATE = True
ADAPTIVE_NEIGHBORS = False
FIX_ATTENTION = True

TOTAL_EPOCHS = 1000
STOP_AT_EPOCH = TOTAL_EPOCHS
EPOCHS_PER_RUN = 100
PRUNE_INTERMEDIATE_CHECKPOINTS = True
KEEP_EVERY = 100
OUTPUT_DIR = DRIVE_ROOT / "GATraj-RiskMotion" / "runs"

## Train or automatically resume from Drive

After a Colab restart, mount the same Drive and rerun the setup, configuration and training cells with the same settings. No manual checkpoint copying is needed. The latest completed epoch is found in the selected experiment directory, and its model, optimizer, scheduler and random-number-generator states are restored.

Only completed checkpoints are selected; temporary checkpoint files are ignored. A runtime disconnect can interrupt an unfinished epoch or a Drive write, so it cannot preserve work that was never saved.

In [ ]:
from train import get_parser, experiment_name

assert DRIVE_ROOT.is_dir(), "Mount Google Drive before training."
assert 0 < STOP_AT_EPOCH <= TOTAL_EPOCHS
assert EPOCHS_PER_RUN > 0 and KEEP_EVERY > 0
common_args = [
    "--test_set", str(TEST_SET), "--seed", str(SEED),
    "--num_epochs", str(TOTAL_EPOCHS),
    "--x_encoder_layers", "3", "--eta_min", "1e-5", "--batch_size", "32",
    "--learning_rate", "5e-4", "--randomRotate", "True", "--final_mode", "20",
    "--neighbor_thred", "10", "--using_cuda", "True", "--gpu", "0", "--clip", "1",
    "--pass_time", "2", "--ifGaussian", "False", "--SR", "True", "--input_offset", "True",
    "--fix_attention", str(FIX_ATTENTION), "--motion_module", str(MOTION_MODULE),
    "--relative_velocity", str(RELATIVE_VELOCITY), "--closest_approach", str(CLOSEST_APPROACH),
    "--confidence_gate", str(CONFIDENCE_GATE), "--adaptive_neighbors", str(ADAPTIVE_NEIGHBORS),
    "--save_base_dir", str(OUTPUT_DIR),
]
run_name = experiment_name(get_parser().parse_args(common_args))
common_args += ["--train_model", run_name]
run_dir = OUTPUT_DIR / str(TEST_SET) / run_name

def checkpoint_files():
    checkpoints = {}
    for path in run_dir.glob(f"{run_name}_*.tar"):
        suffix = path.stem.rsplit("_", 1)[1]
        if suffix.isdigit() and int(suffix) > 0:
            checkpoints[int(suffix)] = path
    return checkpoints

def latest_epoch():
    return max(checkpoint_files(), default=0)

def prune_checkpoints(latest):
    if PRUNE_INTERMEDIATE_CHECKPOINTS:
        for epoch, path in checkpoint_files().items():
            if epoch < latest and epoch % KEEP_EVERY != 0:
                path.unlink()

completed = latest_epoch()
if completed > TOTAL_EPOCHS:
    raise ValueError("This directory contains a checkpoint beyond TOTAL_EPOCHS; check the experiment settings.")
print("Experiment:", run_name)
print("Persistent checkpoint directory:", run_dir)
print("Resuming after epoch:", completed)
while completed < STOP_AT_EPOCH:
    count = min(EPOCHS_PER_RUN, STOP_AT_EPOCH - completed)
    subprocess.run([sys.executable, "train.py", *common_args,
                    "--epochs_per_run", str(count), "--load_model", str(completed)], check=True)
    advanced = latest_epoch()
    if advanced != completed + count:
        raise RuntimeError("Expected checkpoint was not saved; inspect the training output.")
    completed = advanced
    prune_checkpoints(completed)
    print("Saved to Drive through epoch", completed)
print("Completed through epoch", completed)

## Evaluate the predetermined final checkpoint

The report contains minADE20/minFDE20, observed-density subsets, parameter count, synchronized forward timing and a top-1 collision diagnostic. Use the same GPU for timing comparisons. Evaluate the predetermined final epoch for the comparison; early checkpoints are startup checks rather than a basis for selecting the best test result.

In [ ]:
import json

assert latest_epoch() >= TOTAL_EPOCHS, "Finish the predetermined training horizon before the final comparison."
subprocess.run([sys.executable, "evaluate.py", *common_args,
                "--phase", "test", "--load_model", str(TOTAL_EPOCHS)], check=True)
report_path = run_dir / f"evaluation_{TOTAL_EPOCHS}.json"
report = json.loads(report_path.read_text())
print(json.dumps({key: report[key] for key in ("groups", "parameters", "device", "forward_ms_per_batch")}, indent=2))
print("Report saved in Drive:", report_path)

## Complete the comparison

Repeat for all five `TEST_SET` values, using the same seeds in baseline and modified runs. Use seeds 1, 2 and 3 where resources permit. Average the five folds equally within each seed, then report mean and standard deviation across seeds. Component ablations are listed in the repository README and `docs/EXPERIMENTS.md`.

All persistent files are under `MyDrive/GATraj-RiskMotion/runs/<fold>/<experiment>/`. The default cleanup retains every 100th checkpoint plus the latest, and leaves configurations and evaluation reports intact. Keep the final checkpoint for each experiment.